# Phương pháp 4: Phân tích Loss Landscape qua Bài toán Teacher-Student

## Mục tiêu

Với mô hình đơn giản $h_{t+1} = \lambda h_t + x_t$, chúng ta sẽ:
1. Cố định một **teacher** với $\lambda^* \in [0, 1)$
2. Train một **student** để bắt chước teacher
3. Trực quan hóa loss landscape

Khi $\lambda^* \to 1$ (memory dài), loss landscape trở nên **sắc nhọn** và **không lồi**, làm cho gradient-based optimization gần như bất khả thi.

## Setup teacher-student

**Teacher**: $h_{t+1}^* = \lambda^* h_t^* + x_t$, đầu ra $y_t = c \cdot h_t^*$

**Student**: dự đoán $\hat{y}_t = c \cdot h_t$ với $h_{t+1} = \lambda h_t + x_t$

**Loss**: $L(\lambda) = \frac{1}{T}\sum_{t=1}^{T}(y_t - \hat{y}_t)^2$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm

np.random.seed(7)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Bước 1: Bài toán 1D — loss landscape theo λ

In [ ]:
def run_rnn(lam, x):
    T = len(x)
    h = np.zeros(T + 1)
    for t in range(T):
        h[t+1] = lam * h[t] + x[t]
    return h[1:]

def loss_curve(lam_star, lam_arr, T=300, n_avg=20):
    losses = np.zeros(len(lam_arr))
    for _ in range(n_avg):
        x = np.random.randn(T)
        y_teacher = run_rnn(lam_star, x)
        for i, lam in enumerate(lam_arr):
            y_student = run_rnn(lam, x)
            losses[i] += np.mean((y_teacher - y_student) ** 2)
    return losses / n_avg

In [ ]:
lam_arr = np.linspace(-0.99, 0.99, 401)
teachers = [0.3, 0.7, 0.9, 0.99]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, lam_star in zip(axes.flat, teachers):
    L = loss_curve(lam_star, lam_arr, T=200, n_avg=30)
    ax.plot(lam_arr, L, 'b-', linewidth=2)
    ax.axvline(lam_star, color='red', linestyle='--', label=f'λ* = {lam_star}')
    ax.set_yscale('log')
    ax.set_xlabel('λ (student)')
    ax.set_ylabel('Loss')
    ax.set_title(f'Loss landscape, λ* = {lam_star}')
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle('Loss càng "sắc nhọn" khi λ* càng gần 1 (memory dài)', y=1.01)
plt.tight_layout()
plt.show()

**Quan sát**: Khi $\lambda^* \to 1$:
- Đáy hàm loss trở nên **rất hẹp** (sharp minimum)
- Độ dốc tại nghiệm tối ưu **lớn vô cùng**
- Đây là minh chứng trực quan cho "curse of memory"

## Bước 2: Zoom vào vùng quanh nghiệm tối ưu — độ cong

In [ ]:
lam_zoom = np.linspace(-0.05, 0.05, 401)
fig, ax = plt.subplots(figsize=(11, 6))
for lam_star in teachers:
    lam_arr_zoom = lam_star + lam_zoom
    L = loss_curve(lam_star, lam_arr_zoom, T=300, n_avg=50)
    ax.plot(lam_zoom, L, label=f'λ* = {lam_star}', linewidth=2)
ax.set_yscale('log')
ax.set_xlabel('Δλ = λ - λ*')
ax.set_ylabel('Loss (log)')
ax.set_title('Zoom quanh nghiệm tối ưu — đường cong càng dốc khi λ* → 1')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Bước 3: Mô hình 2 chiều (λ, c) — landscape 3D

In [ ]:
def loss_2d(lam_star, c_star, lam_g, c_g, T=200, n_avg=20):
    L = np.zeros_like(lam_g)
    for _ in range(n_avg):
        x = np.random.randn(T)
        y_teacher = c_star * run_rnn(lam_star, x)
        for idx in np.ndindex(lam_g.shape):
            lam = lam_g[idx]; c = c_g[idx]
            y_student = c * run_rnn(lam, x)
            L[idx] += np.mean((y_teacher - y_student) ** 2)
    return L / n_avg

lam_axis = np.linspace(-0.5, 0.999, 50)
c_axis = np.linspace(0.1, 2.0, 50)
Lg, Cg = np.meshgrid(lam_axis, c_axis)

L_easy = loss_2d(0.5, 1.0, Lg, Cg, T=150, n_avg=10)
L_hard = loss_2d(0.95, 1.0, Lg, Cg, T=150, n_avg=10)

In [ ]:
fig = plt.figure(figsize=(15, 6))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(Lg, Cg, np.log10(L_easy + 1e-9), cmap='viridis', alpha=0.85)
ax1.set_xlabel('λ'); ax1.set_ylabel('c'); ax1.set_zlabel('log Loss')
ax1.set_title('λ* = 0.5 — Landscape mượt')

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(Lg, Cg, np.log10(L_hard + 1e-9), cmap='inferno', alpha=0.85)
ax2.set_xlabel('λ'); ax2.set_ylabel('c'); ax2.set_zlabel('log Loss')
ax2.set_title('λ* = 0.95 — Landscape sắc nhọn, khe hẹp')
plt.tight_layout()
plt.show()

In [ ]:
# Contour 2D để thấy rõ hình dạng curse
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, L, title, star in [(axes[0], L_easy, 'λ*=0.5 (smooth)', 0.5),
                             (axes[1], L_hard, 'λ*=0.95 (sharp)', 0.95)]:
    cs = ax.contourf(Lg, Cg, np.log10(L + 1e-9), levels=25, cmap='viridis')
    ax.contour(Lg, Cg, np.log10(L + 1e-9), levels=10, colors='white', linewidths=0.4)
    ax.plot(star, 1.0, 'r*', markersize=18, label='Optimum')
    plt.colorbar(cs, ax=ax)
    ax.set_xlabel('λ'); ax.set_ylabel('c')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Bước 4: SGD trajectory — bằng chứng SGD bất ổn ở λ* gần 1

In [ ]:
def gradient_loss(lam, c, x, y_teacher):
    T = len(x)
    h = np.zeros(T + 1)
    d_lam = np.zeros(T + 1)
    for t in range(T):
        d_lam[t+1] = h[t] + lam * d_lam[t]
        h[t+1] = lam * h[t] + x[t]
    y_pred = c * h[1:]
    err = y_pred - y_teacher
    grad_lam = 2 * np.mean(err * c * d_lam[1:])
    grad_c = 2 * np.mean(err * h[1:])
    loss = np.mean(err ** 2)
    return loss, grad_lam, grad_c

def sgd_train(lam_star, lr=0.01, n_iters=500, T=150, init=(-0.5, 0.5), batch=8):
    lam, c = init
    traj_lam = [lam]; traj_c = [c]; losses = []
    for it in range(n_iters):
        gl = 0; gc = 0; ls = 0
        for _ in range(batch):
            x = np.random.randn(T)
            y_t = 1.0 * run_rnn(lam_star, x)
            l, g_l, g_c = gradient_loss(lam, c, x, y_t)
            gl += g_l / batch; gc += g_c / batch; ls += l / batch
        lam -= lr * gl; c -= lr * gc
        lam = np.clip(lam, -0.999, 0.999)
        traj_lam.append(lam); traj_c.append(c); losses.append(ls)
    return np.array(traj_lam), np.array(traj_c), np.array(losses)

tl1, tc1, ls1 = sgd_train(0.5, lr=0.05, n_iters=300)
tl2, tc2, ls2 = sgd_train(0.95, lr=0.05, n_iters=300)
tl3, tc3, ls3 = sgd_train(0.95, lr=0.005, n_iters=300)
print('Cuối:')
print(f'  λ*=0.5,  lr=0.05  → λ={tl1[-1]:.4f}, c={tc1[-1]:.4f}, loss={ls1[-1]:.6f}')
print(f'  λ*=0.95, lr=0.05  → λ={tl2[-1]:.4f}, c={tc2[-1]:.4f}, loss={ls2[-1]:.6f}')
print(f'  λ*=0.95, lr=0.005 → λ={tl3[-1]:.4f}, c={tc3[-1]:.4f}, loss={ls3[-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
configs = [(L_easy, tl1, tc1, 'λ*=0.5, lr=0.05', 0.5),
           (L_hard, tl2, tc2, 'λ*=0.95, lr=0.05 (UNSTABLE)', 0.95),
           (L_hard, tl3, tc3, 'λ*=0.95, lr=0.005 (slow)', 0.95)]
for ax, (L, tl, tc, title, star) in zip(axes, configs):
    ax.contourf(Lg, Cg, np.log10(L + 1e-9), levels=25, cmap='viridis')
    ax.plot(tl, tc, 'w-', linewidth=1.4, alpha=0.9)
    ax.plot(tl[0], tc[0], 'wo', markersize=8, label='Khởi tạo')
    ax.plot(tl[-1], tc[-1], 'rs', markersize=10, label='Kết thúc')
    ax.plot(star, 1.0, 'r*', markersize=15, label='Optimum')
    ax.set_xlabel('λ'); ax.set_ylabel('c')
    ax.set_title(title); ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ls1, label='λ*=0.5, lr=0.05')
ax.plot(ls2, label='λ*=0.95, lr=0.05')
ax.plot(ls3, label='λ*=0.95, lr=0.005')
ax.set_yscale('log')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss (log)')
ax.set_title('Đường loss SGD — λ*=0.95 bị bật văng hoặc hội tụ rất chậm')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Kết luận phương pháp 4

- **Loss landscape trở nên sắc nhọn** khi $\lambda^* \to 1$ (memory dài).
- **Vùng nghiệm tối ưu** trở thành khe hẹp với độ dốc cực lớn quanh nó.
- **SGD bị bật văng** với learning rate vừa phải, hoặc **hội tụ cực chậm** với lr nhỏ.
- Đây là chứng minh trực quan cho "curse of memory": vấn đề không nằm ở gradient bùng nổ trong runtime, mà ở **hình dạng loss landscape** khi cần ký ức dài.
- Phương pháp tiếp theo (Hessian) sẽ định lượng chính xác độ "sắc nhọn" này.